In [2]:
with open("input.txt", "r") as f:
    text = f.read()

In [3]:
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [4]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(f"Vocabulary size: {vocab_size}")


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
Vocabulary size: 65


In [5]:
# to tokenize the text, we can use the following code:

stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

print(encode("Hello, World!"))
print(decode(encode("Hello, World!")))

[20, 43, 50, 50, 53, 6, 1, 35, 53, 56, 50, 42, 2]
Hello, World!


In [6]:
import torch

data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

In [7]:
# spliting the data into train and a validation split.

n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

In [8]:
block_size = 8 # how many characters to consider for predictions (max context length)
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [9]:
x = train_data[:block_size]
y = train_data[1:block_size+1]

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target: {target}")

when input is tensor([18]) the target: 47
when input is tensor([18, 47]) the target: 56
when input is tensor([18, 47, 56]) the target: 57
when input is tensor([18, 47, 56, 57]) the target: 58
when input is tensor([18, 47, 56, 57, 58]) the target: 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target: 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target: 58


In [10]:
torch.manual_seed(1377)
batch_size = 4 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('-'*15)

for b in range(batch_size): # batch dimention
    for t in range(block_size): # time dimention
        context = xb[b, :t+1]
        target = yb[b, t]
        print(f"when input is {context.tolist()} the target: {target}")

inputs:
torch.Size([4, 8])
tensor([[58, 46, 43,  1, 58, 47, 51, 43],
        [50, 50,  1, 51, 43,  1, 52, 53],
        [57,  1, 49, 52, 53, 58,  1, 49],
        [59, 45, 46, 58, 43, 56,  8,  0]])
targets:
torch.Size([4, 8])
tensor([[46, 43,  1, 58, 47, 51, 43,  1],
        [50,  1, 51, 43,  1, 52, 53, 58],
        [ 1, 49, 52, 53, 58,  1, 49, 52],
        [45, 46, 58, 43, 56,  8,  0, 35]])
---------------
when input is [58] the target: 46
when input is [58, 46] the target: 43
when input is [58, 46, 43] the target: 1
when input is [58, 46, 43, 1] the target: 58
when input is [58, 46, 43, 1, 58] the target: 47
when input is [58, 46, 43, 1, 58, 47] the target: 51
when input is [58, 46, 43, 1, 58, 47, 51] the target: 43
when input is [58, 46, 43, 1, 58, 47, 51, 43] the target: 1
when input is [50] the target: 50
when input is [50, 50] the target: 1
when input is [50, 50, 1] the target: 51
when input is [50, 50, 1, 51] the target: 43
when input is [50, 50, 1, 51, 43] the target: 1
when inpu

In [11]:
print(xb)

tensor([[58, 46, 43,  1, 58, 47, 51, 43],
        [50, 50,  1, 51, 43,  1, 52, 53],
        [57,  1, 49, 52, 53, 58,  1, 49],
        [59, 45, 46, 58, 43, 56,  8,  0]])


In [17]:
# bigram model is the simplest possible language model, it just looks at the last character to predict the next character. 
# We can implement this as a simple lookup table of logits for each possible pair of characters.

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(1377)

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) # (Batch ,Time ,Channels)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)

            loss = F.cross_entropy(logits, targets) # basically we want to measure the loss between the predicted logits and the actual targets. We use cross-entropy loss for this purpose.
            # pytorch's cross_entropy function expects the input to be of shape (N, C) where N is the number of samples and C is the number of classes. So we reshape the logits and targets to be of shape (B*T, vocab_size) and (B*T,) respectively.

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx, targets=None)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx


m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)


idx = torch.zeros((1, 1), dtype=torch.long) # starting with a batch of size 1 and a sequence length of 1
print(decode(m.generate(idx, max_new_tokens=100)[0].tolist())) # generate 100 new characters from the model

torch.Size([32, 65])
tensor(4.8616, grad_fn=<NllLossBackward0>)

;yV:HgfkJsDhl
GxB$bskAAH3c,lNe!rjsKqZNX:KeFKNTU,q xFWOdV.DS,bR'TRhlU ioazLx,gCdhAWjvOVUxpjFPrp'Wm
C



In [18]:
# making a optimizer to train the model. We will use AdamW optimizer which is a variant of Adam optimizer with weight decay.
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [46]:
batch_size = 32
for steps in range(10000):
    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(f"step {steps}: loss {loss.item()}")

step 9999: loss 2.3199548721313477


In [47]:
print(decode(m.generate(idx, max_new_tokens=500)[0].tolist())) # generate 100 new characters from the model



Sir merads

Henganend dewisedico t athemamy'dos geeiallt hy.
ARLAsho'lly,
RDeroularpow ad e.

H:

Dule hivee beghe be bjothinemy irje Yom thanndorthe ooto t h ld iomor lyour r bril uste, dible? Gorivy heatowenisot  t andlinis I me t s el; l itoofrwhele.
catizey s.
ANon hathe d

Han, che hal y my CLI fofacak; sp--nwheefod s al pr wil fonchinoumpe at t me biratho hatof pr cakicee:

BO: hisinomuitlersentires BHA olllll tis.
Wacel t Hobu
TAse b--r, f:
'se lleeithet? ounthatid han fa te knghon l'la 


## The Mathematical Trick behind the Self Attention Principle.

In [63]:
torch.manual_seed(1337)
B, T, C = 4, 8, 2
x = torch.randn(B, T, C)
x.shape

torch.Size([4, 8, 2])

In [64]:
# we basically want the token with itself and the previous token but not the once in the future.
# so we want: x[b, t] = mean{i<=t} x[b, i]
xbow = torch.zeros((B, T, C)) # x bag of words
for b in range(B):
    for t in range(T):
        xprev = x[b, :t+1] # (t+1, C)
        xbow[b, t] = torch.mean(xprev, 0) # (C)


In [51]:
x[0]

tensor([[ 0.1808, -0.0700],
        [-0.3596, -0.9152],
        [ 0.6258,  0.0255],
        [ 0.9545,  0.0643],
        [ 0.3612,  1.1679],
        [-1.3499, -0.5102],
        [ 0.2360, -0.2398],
        [-0.9211,  1.5433]])

In [52]:
xbow[0]

tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])

As we can see above the first cell is same for x and xbow but the subsiquent once are different as xbow has the averages of the values present unlike x

---
now we need to make the code optimized. so we will use matrix.


In [ ]:
# example
torch.manual_seed(42)
a = torch.ones((3, 3))
a_tril = torch.tril(a) # this is done to make sure that we only consider the current and previous tokens and not the future tokens. The lower triangular matrix will have 1s in the lower triangle and 0s in the upper triangle.
a = a_tril / torch.sum(a_tril, dim=1, keepdim=True) # this is done to make sure that the sum of the weights is 1. We divide each row by the sum of the row.
b = torch.randint(0, 10, (3, 2)).float()
c = a @ b
# by manipulating the multiplication of the lower triangular matrix with the input tensor, we can achieve the desired effect of only considering the average of current and previous tokens and not the future tokens.
print('a=')
print(a)
print('b=')
print(b)
print('c=')
print(c)

a=
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
b=
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
c=
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


In [77]:
# version 2
# now making the above example from a for loop to a single matrix multiplication. 
# We can do this by creating a lower triangular matrix of 1s and then normalizing it so that the sum of each row is 1. 
# Then we can multiply this matrix with the input tensor to get the desired output.

wei = torch.tril(torch.ones(T, T)) # (T, T)
wei = wei / wei.sum(1, keepdim=True) # (T, T)
xbow2 = wei @ x # (B, T, T) @ (B, T, C) ---> (B, T, C) making xbow2 = xbow



In [78]:
# version 3 using softmax
tril = torch.tril(torch.ones(T, T)) # (T, T)
wei = torch.zeros((T, T))
wei = wei.masked_fill(tril == 0, float('-inf')) # makes the values in the upper triangular part of the matrix to be -inf so that when we apply softmax, they will become 0. This is done to make sure that we only consider the current and previous tokens and not the future tokens.
wei = F.softmax(wei, dim=-1) # (T, T)
# soft max basically takes the exponent of each element in the matrix and then normalizes it by dividing by the sum of the exponentials. This ensures that the sum of each row is 1 and all values are positive. The softmax function is often used in machine learning to convert logits into probabilities.

xbow3 = wei @ x # (B, T, T) @ (B, T, C) ---> (B, T, C) making xbow3 = xbow
torch.allclose(xbow2, xbow3) # this is done to make sure that the two methods give the same result. The allclose function checks if the two tensors are equal within a certain tolerance.

True

In [90]:
# version 4 : self attention head

torch.manual_seed(1337)
B, T, C = 4, 8 , 32
x = torch.randn(B, T, C)

#implementing a single head of self attention. The idea is to compute the attention weights for each token in the sequence based on its similarity to all other tokens in the sequence. We can do this by computing the dot product of the query and key vectors for each token, applying a softmax to get the attention weights, and then using these weights to compute a weighted sum of the value vectors for each token.
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
k = key(x) # (B, T, head_size)
q = query(x) # (B, T, head_size)

wei= q @ k.transpose(-2, -1) #(B, T, head_size) @ (B, head_size, T) ---> (B, T, T)



tril = torch.tril(torch.ones(T, T)) # (T, T)
# wei = torch.zeros((T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1) # (T, T)

v = value(x)
out = wei @ v
# out = wei @ x # (B, T, T) @ (B, T, C) ---> (B, T, C)

out.shape

torch.Size([4, 8, 16])

In [89]:
wei[0]

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1574, 0.8426, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2088, 0.1646, 0.6266, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5792, 0.1187, 0.1889, 0.1131, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0294, 0.1052, 0.0469, 0.0276, 0.7909, 0.0000, 0.0000, 0.0000],
        [0.0176, 0.2689, 0.0215, 0.0089, 0.6812, 0.0019, 0.0000, 0.0000],
        [0.1691, 0.4066, 0.0438, 0.0416, 0.1048, 0.2012, 0.0329, 0.0000],
        [0.0210, 0.0843, 0.0555, 0.2297, 0.0573, 0.0709, 0.2423, 0.2391]],
       grad_fn=<SelectBackward0>)

Notes:
- Attention is a **communication mechanism**. Can be seen as nodes in a directed graph looking at each other and aggregating information with a weighted sum from all nodes that point to them, with data-dependent weights.
- There is no notion of space. Attention simply acts over a set of vectors. This is why we need to positionally encode tokens.
- Each example across batch dimension is of course processed completely independently and never "talk" to each other
- In an "encoder" attention block just delete the single line that does masking with `tril`, allowing all tokens to communicate. This block here is called a "decoder" attention block because it has triangular masking, and is usually used in autoregressive settings, like language modeling.
- "self-attention" just means that the keys and values are produced from the same source as queries. In "cross-attention", the queries still get produced from x, but the keys and values come from some other, external source (e.g. an encoder module)
- "Scaled" attention additional divides `wei` by 1/sqrt(head_size). This makes it so when input Q,K are unit variance, wei will be unit variance too and Softmax will stay diffuse and not saturate too much. Illustration below

In [96]:
k = torch.randn(B, T, head_size) # (B, T, head_size)
q = torch.randn(B, T, head_size) # (B, T, head_size)
wei = q @ k.transpose(-2, -1) * head_size**(-0.5) #(B, T, head_size) @ (B, head_size, T) ---> (B, T, T)

In [97]:
k.var()

tensor(0.9006)

In [98]:
q.var()

tensor(1.0037)

In [99]:
wei.var()

tensor(0.9957)

## Batch normalization vs Layer normalization

In [100]:
class LayerNorm1d: # (used to be BatchNorm1d)

  def __init__(self, dim, eps=1e-5, momentum=0.1):
    self.eps = eps
    self.gamma = torch.ones(dim)
    self.beta = torch.zeros(dim)

  def __call__(self, x):
    # calculate the forward pass
    # for batch normalization, we would calculate the mean and variance across the batch dimension (dim=0). However, for layer normalization, we calculate the mean and variance across the feature dimension (dim=1). This is because layer normalization normalizes each sample independently, rather than normalizing across the batch. 
    xmean = x.mean(1, keepdim=True) # batch mean = 1
    xvar = x.var(1, keepdim=True) # batch variance = 1
    xhat = (x - xmean) / torch.sqrt(xvar + self.eps) # normalize to unit variance
    self.out = self.gamma * xhat + self.beta
    return self.out

  def parameters(self):
    return [self.gamma, self.beta]

torch.manual_seed(1337)
module = LayerNorm1d(100)
x = torch.randn(32, 100) # batch size 32 of 100-dimensional vectors
x = module(x)
x.shape

torch.Size([32, 100])